In [2]:
import pickle
from generator import graph_generator, deterministic_ds
import os
import sys
sys.path.append("..")
from helpers.tools import draw_nx, to_nx_graph
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from data.generator import load_deterministic_ds

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import statsmodels.tsa.vector_ar

In [5]:
def shuffle_links(d):
    a = torch.randperm(d[1].shape[0])
    b = torch.randperm(d[1].shape[1])
    c = torch.randperm(d[1].shape[2])
    d_random = d[1][a][:,b][:,:,c]
    d = (d[0], d_random)
    return d

In [6]:
def make_deterministic_ds(
        epoch_size = 300,
        n_vars = 3,
        max_lags = 2,
        link_threshold = 0.85,
        param_range = [0.3,0.5],
        test_range = [0.2,0.3],
        val_range = [0.2, 0.3],
        restrict_links = True,
        noise_var = 0.4,
        noise_test = 0.6, 
        noise_val = 0.4,
        time_series_n = 500,
        binary = False,
        tr_n= 5000,
        te_n= 500,
        eval_n = 500,
        name = "base_train",
        min_max_scaling = True,
        threshold_filter = [-10,10],
        random = False,
        variable_noise = False,
        functional_class = "linear", 
        mirror_range = False,
        n_fixed_labels = 0, 
        distinguish_mode = False,
    ):
    
    p = "deterministic_ds/" + name
    isExist = os.path.exists(p)
    if not isExist:
        os.makedirs(p)

        pickle.dump(locals(), open(p + "/args.p", "wb"))

        A = graph_generator(
                epoch_size = epoch_size,
                n_vars = n_vars,
                max_lags = max_lags,
                link_threshold = link_threshold,
                param_range = param_range,
                restrict_links = restrict_links,
                noise_var = noise_var,
                time_series_n = time_series_n,
                binary = binary,
                min_max_scaling=min_max_scaling,
                threshold_filter=threshold_filter,
                variable_noise=variable_noise,
                functional_class= functional_class,
                mirror_range= mirror_range,
                return_nonlinear= True,
                n_fixed_labels = n_fixed_labels,
        )
        ds =  []
        ds_val = []
        ds_test = []

        # Check whether the specified path exists or not
        if distinguish_mode:
            for x in range(int(tr_n / 2)):
                d = A.__getitem__(10)
                ds.append(d)
            A.param_range = val_range
            A.noise_var =noise_val
            for x in range(int(eval_n / 2)): 
                d = A.__getitem__(10)
                ds_val.append(d)
            A.param_range = test_range
            A.noise_var =noise_test
            for x in range(te_n): 
                d = A.__getitem__(10)
                ds_test.append(d)
            A.param_range = param_range
            A.noise_var =noise_var
            print("changing functional class")
            A.functional_class = "nonlinear"
            A.nonlinear_threshold = 1
            A.func_options = A.init_func_options()
            for x in range(int(tr_n / 2)):
                d = A.__getitem__(10)
                ds.append(d)
            A.param_range = val_range
            A.noise_var =noise_val
            for x in range(int(eval_n / 2)): 
                d = A.__getitem__(10)
                ds_val.append(d)
            A.param_range = test_range
            A.noise_var =noise_test
            for x in range(te_n): 
                d = A.__getitem__(10)
                ds_test.append(d)


        else:
            for x in range(tr_n):
                d = A.__getitem__(10)
                if random:
                    # shuffle the links
                    d = shuffle_links(d)
                ds.append(d)

            A.param_range = val_range
            A.noise_var =noise_val
            for x in range(eval_n): 
                d = A.__getitem__(10)
                if random:
                    # shuffle the links
                    d = shuffle_links(d)
                ds_val.append(d)

            A.param_range = test_range
            A.noise_var =noise_test
            for x in range(te_n): 
                d = A.__getitem__(10)
                if random:
                    # shuffle the links
                    d = shuffle_links(d)
                ds_test.append(d)

        pickle.dump(ds,open(p + "/simple_train.p", "wb"))
        pickle.dump(ds_val,open(p + "/simple_val.p", "wb"))
        pickle.dump(ds_test,open(p + "/simple_test.p", "wb"))
    else:
        print("Dataset exists. Please choose a different name.")

    #python program to check if a directory exists

In [10]:
# Large var set

make_deterministic_ds(name="scale_up",
                    max_lags=3,
                    n_vars=10,
                    test_range=[0.2,0.3],
                    val_range=[0.2,0.3],
                    noise_test= 0.6,
                    mirror_range=True,
                    link_threshold=0.9)
# Large var set

make_deterministic_ds(name="scale_up_additional",
                    max_lags=3,
                    n_vars=10,
                    test_range=[0.3,0.5],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    mirror_range=True,
                    link_threshold=0.9)

In [11]:
# Large var set

make_deterministic_ds(name="scale_up_2",
                    max_lags=3,
                    n_vars=7,
                    test_range=[0.2,0.3],
                    val_range=[0.2,0.3],
                    noise_test= 0.6,
                    mirror_range=True,
                    link_threshold=0.9)
# Large var set

make_deterministic_ds(name="scale_up_additional_2",
                    max_lags=3,
                    n_vars=7,
                    test_range=[0.3,0.5],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    mirror_range=True,
                    link_threshold=0.9)

In [12]:
# Large var set

make_deterministic_ds(name="scale_up_3",
                    max_lags=3,
                    n_vars=15,
                    test_range=[0.2,0.3],
                    val_range=[0.2,0.3],
                    noise_test= 0.6,
                    mirror_range=True,
                    link_threshold=0.9)
# Large var set

make_deterministic_ds(name="scale_up_additional_3",
                    max_lags=3,
                    n_vars=15,
                    test_range=[0.3,0.5],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    mirror_range=True,
                    link_threshold=0.9)

In [7]:
# Large var set

make_deterministic_ds(name="scale_up_4",
                    max_lags=3,
                    n_vars=12,
                    test_range=[0.2,0.3],
                    val_range=[0.2,0.3],
                    noise_test= 0.6,
                    mirror_range=True,
                    link_threshold=0.9)
# Large var set

make_deterministic_ds(name="scale_up_additional_4",
                    max_lags=3,
                    n_vars=12,
                    test_range=[0.3,0.5],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    mirror_range=True,
                    link_threshold=0.9)

# Experiment set 1

In [30]:
make_deterministic_ds(name="uncover_params_small",
                    max_lags=2,
                    n_vars=3,
                    test_range=[0.2,0.3],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    tr_n=500,
                    link_threshold=0.5,
                    n_fixed_labels=1)

save label


In [26]:
make_deterministic_ds(name="uncover_params",
                    max_lags=3,
                    n_vars=5,
                    test_range=[0.2,0.3],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    tr_n=500,
                    link_threshold=0.5
                    n_fixed_labels=1)

Dataset exists. Please choose a different name.


In [27]:
make_deterministic_ds(name="classify_graphs",
                    max_lags=2,
                    n_vars=3,
                    test_range=[0.2,0.3],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    tr_n=500,
                    link_threshold=0.5,
                    n_fixed_labels=5)

save label
save label
save label
save label
save label


In [28]:
make_deterministic_ds(name="distinguish",
                    max_lags=2,
                    n_vars=3,
                    test_range=[0.2,0.3],
                    val_range=[0.3,0.5],
                    noise_test= 0.4,
                    tr_n=500,
                    n_fixed_labels=1,
                    distinguish_mode=True)

save label
changing functional class


# The 6 sets

In [7]:
make_deterministic_ds(name="base_additional",
                    max_lags=2,
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,
                    n_vars=3)

In [8]:
make_deterministic_ds(name="big_base_additional",
                    max_lags=3,
                    n_vars=5, 
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,
                    link_threshold=0.9
                    )

In [9]:
make_deterministic_ds(name="big_base_mirror_additional",
                    max_lags=3,
                    n_vars=5, 
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,
                    mirror_range=True, 
                    link_threshold = 0.9
                    )

In [15]:
make_deterministic_ds(name="poly_additional",
                    functional_class="simple_nonlinear",
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,)

In [11]:
make_deterministic_ds(name="big_poly_additional",
                    max_lags=3,
                    n_vars=5, 
                    mirror_range=True,
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,
                    link_threshold = 0.9,
                    functional_class="simple_nonlinear")

In [12]:
make_deterministic_ds(name="nl_additional",
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,
                    functional_class="nonlinear")

In [13]:
make_deterministic_ds(name="big_nl_additional",
                    max_lags=3,
                    n_vars=5, 
                    test_range= [0.3,0.5],
                    val_range= [0.3,0.5],
                    noise_test= 0.4,
                    mirror_range=True,
                    link_threshold = 0.9,
                    functional_class="nonlinear")

# Join everything I have together for the big dataset

In [1]:
from os import listdir
from os.path import isfile, join
mypath = "deterministic_ds"
onlyfiles = [mypath + "/" + f + "/" for f in listdir(mypath)]

In [5]:
onlyfiles = []
for x in ["big_base_mirror", "big_nl", "big_poly", "poly", "nl", "base"]:
    onlyfiles.append("deterministic_ds/" + x + "/")
    onlyfiles.append("deterministic_ds/" + x + "_additional/")

In [12]:
stack = [] # train
stack2 = [] # val out of range
stack3 = [] # test out of range
stack4 = [] # test2 out of range
for f in onlyfiles: 
    if "additional" in f:
        stack.append(pickle.load(open(f + "simple_train.p", "rb")))
        stack.append(pickle.load(open(f + "simple_val.p", "rb")))
        stack4.append(pickle.load(open(f + "simple_test.p", "rb")))
    else:
        stack.append(pickle.load(open(f + "simple_train.p", "rb")))
        stack2.append(pickle.load(open(f + "simple_val.p", "rb")))
        stack3.append(pickle.load(open(f + "simple_test.p", "rb")))

stack = [item for sublist in stack for item in sublist]
stack2 = [item for sublist in stack2 for item in sublist]
stack3 = [item for sublist in stack3 for item in sublist]
stack4 = [item for sublist in stack4 for item in sublist]

In [13]:
len(stack), len(stack2), len(stack3), len(stack4)

(63000, 3000, 3000, 3000)

In [14]:
for group in [stack,stack2,stack3, stack4]:
    for x in range(len(group)):
        if group[x][0].shape[1] == 3:
            a= torch.concat([group[x][0], torch.normal(0, 0.01, (500,2))], axis=1)
            b = torch.nn.functional.pad(group[x][1], (1,0,0,2,0,2), mode = "constant", value = 0.0)
            group[x] = (a,b)

In [16]:
len(stack), len(stack2), len(stack3), len(stack4)

(63000, 3000, 3000, 3000)

In [17]:
pickle.dump(stack,open("deterministic_ds/joint/simple_train.p", "wb"))
pickle.dump(stack2,open("deterministic_ds/joint/simple_val.p", "wb"))
pickle.dump(stack3,open("deterministic_ds/joint/simple_test.p", "wb"))

In [18]:
pickle.dump(stack,open("deterministic_ds/joint_additional/simple_train.p", "wb"))
pickle.dump(stack2,open("deterministic_ds/joint_additional/simple_val.p", "wb"))
pickle.dump(stack4,open("deterministic_ds/joint_additional/simple_test.p", "wb"))